## Implement MLFlow for House Price Prediction

1. Run a hyperparameter tuning while training a model
2. Log every Hyperparamter and metrics in the MLFlow UI
3. Compare the results of the various runs in the MLFlow UI
4. Choose the best run and register it as a model

### Installing Dependencies

In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.datasets import fetch_california_housing


In [2]:
housing = fetch_california_housing()
housing

{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
           37.88      , -122.23      ],
        [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
           37.86      , -122.22      ],
        [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
           37.85      , -122.24      ],
        ...,
        [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
           39.43      , -121.22      ],
        [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
           39.43      , -121.32      ],
        [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
           39.37      , -121.24      ]], shape=(20640, 8)),
 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,)),
 'frame': None,
 'target_names': ['MedHouseVal'],
 'feature_names': ['MedInc',
  'HouseAge',
  'AveRooms',
  'AveBedrms',
  'Population',
  'AveOccup',
  'Latitude',
  'Longitude'],
 'DESCR': 

### Preparing the dataset

In [4]:
data = pd.DataFrame(housing.data, columns=housing.feature_names)
data["Price"] = housing.target
data.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


### Train Test Split, Hyperparameters

In [5]:
from urllib.parse import urlparse

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(data.drop(["Price"], axis=1),
                                                    data["Price"], 
                                                    test_size=0.2, 
                                                    random_state=42)


In [8]:
# Set a schema for the Inputs and Outputs of the model using mlflow.models.signature
from mlflow.models.signature import infer_signature
infer_signature(X_train, y_train)

inputs: 
  ['MedInc': double (required), 'HouseAge': double (required), 'AveRooms': double (required), 'AveBedrms': double (required), 'Population': double (required), 'AveOccup': double (required), 'Latitude': double (required), 'Longitude': double (required)]
outputs: 
  ['Price': double (required)]
params: 
  None

#### Define hyperparameter Grid

In [9]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

In [10]:
## Performing Hyperparameter Tuning using GridSearchCV

def hyperparameter_tuning(X_train, y_train, param_grid):
    rf = RandomForestRegressor(random_state=42)
    grid_search = GridSearchCV(estimator = rf,
                               param_grid = param_grid,
                               cv = 3,
                               n_jobs = -1,
                               verbose = 2,
                               scoring = "neg_mean_squared_error")
    
    grid_search.fit(X_train, y_train)
    return grid_search

#### Start the MLFlow Experiment

In [11]:
with mlflow.start_run(run_name="RandomForestRegressor_Hyperparameter_Tuning"):

    # Perform hyperparameter tuning
    grid_search = hyperparameter_tuning(X_train, y_train, param_grid)

    # Get the best hyperparameters
    best_model = grid_search.best_estimator_

    # Evaluate the best model on the test set
    y_pred = best_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)

    # Log the best hyperparameters and the MSE to MLflow
    mlflow.log_params("best_n_estimators", grid_search.best_params_["n_estimators"])
    mlflow.log_params("best_max_depth", grid_search.best_params_["max_depth"])
    mlflow.log_params("best_min_samples_split", grid_search.best_params_["min_samples_split"])
    mlflow.log_params("best_min_samples_leaf", grid_search.best_params_["min_samples_leaf"])
    mlflow.log_metric("mse", mse)

    # Tracking URI
    mlflow.set_tracking_uri(uri="http://127.0.0.1:5000/")
    tracking_url_type_store = urlparse(mlflow.get_tracking)

    if tracking_url_type_store.scheme != "file":
        mlflow.sklearn.log_model(best_model, "model", registered_model_name="Best Random Forest model")

    else:
        mlflow.sklearn.log_model(best_model, "model", signature = infer_signature(X_train, y_train))


    print(f"Best Hyperparameters: {grid_search.best_params_}")
    print(f"Mean Squared Error: {mse}")

    


Fitting 3 folds for each of 24 candidates, totalling 72 fits


AttributeError: 'str' object has no attribute 'items'